# P2.6 — P2.6 scoring pass — confirmation seed 3407

Scores the held-out DIV-P2-001 windows with the four seed-3407 experts (same windows, envelopes and sanity-reproduction guard as seed 42). Run it after the four training notebooks of seed 3407 have finished.

**Seed 42 stays the primary seed.** This notebook never writes to the seed-42 folders and changes nothing in P2-FREEZE-001 or Phase 3. Protocol: `docs/protocol/SEED-CONFIRM-001-seed-confirmation.md` (also inside the code ZIP).

## Setup

1. In Colab, choose **Runtime → Change runtime type** and select a GPU with **native BF16** and at least 20 GiB (L4, A100, H100 or G4).
2. Upload `seqlogad-code-seeds-v1.zip` once to `MyDrive/SeqLogAD/Seeds/uploads/` (both seeds share it). The data ZIPs are read from the seed-42 upload folders (`MyDrive/SeqLogAD/P2.2_Sequence/uploads/`, `MyDrive/SeqLogAD/P2.1_Semantic/uploads/`, `MyDrive/SeqLogAD/P2.4_GTAT/uploads/`); no second upload is needed.
3. Keep the Hugging Face token in **Colab Secrets** as `HF_TOKEN` with notebook access. Never paste the token into a cell.
4. Choose **Runtime → Run all**.

Outputs go to `MyDrive/SeqLogAD/Seeds/S3407/P2.6/`. If Colab disconnects, run the notebook again: finished parts are skipped and an unfinished fold resumes from its last recovery checkpoint. Disconnect only after `SAFE_TO_DISCONNECT_COLAB=YES`.


In [ ]:
# HUMAN CONFIG — seed/fold are fixed by the development protocol.
from pathlib import Path
import os, uuid
from google.colab import drive

drive.mount("/content/drive")

TASK_ID = "P2.6"
NOTEBOOK_STATUS = "READY"
BLOCKERS = []
RUN_MODE = "development"
SEED = 3407  # SEED-CONFIRM-001 confirmation seed; seed 42 stays primary
TARGETS = ["BGL", "HDFS"]  # EXC-007: Hadoop/OpenStack are source-only
DRIVE_ROOT = Path("/content/drive/MyDrive/SeqLogAD/Seeds/S3407/P2.6")  # this seed's own folder
UPLOAD_ROOT = Path("/content/drive/MyDrive/SeqLogAD/Seeds/uploads")  # the seed code ZIP, shared by both seeds
# Data ZIPs come from the seed-42 upload folders; nothing is written there.
UPLOAD_ROOTS = [UPLOAD_ROOT] + [Path("/content/drive/MyDrive/SeqLogAD") / folder / "uploads" for folder in ("P2.2_Sequence", "P2.1_Semantic", "P2.4_GTAT",)]
SEQLOGAD = Path("/content/drive/MyDrive/SeqLogAD")
FINAL_ZIPS = {
    "P2.1": SEQLOGAD / "Seeds/S3407/P2.1/seqlogad_outputs_P2.1_S3407_FINAL.zip",
    "P2.2": SEQLOGAD / "Seeds/S3407/P2.2/seqlogad_outputs_P2.2_S3407_FINAL.zip",
    "P2.3": SEQLOGAD / "Seeds/S3407/P2.3/seqlogad_outputs_P2.3_S3407_FINAL.zip",
    "P2.4": SEQLOGAD / "Seeds/S3407/P2.4/seqlogad_outputs_P2.4_S3407_FINAL.zip",
}
SCORING_TASKS = "P2.4,P2.3,P2.2,P2.1"
PERSISTENT_ROOT = DRIVE_ROOT
# Keep Python, extracted source, tokenizer cache and temporary outputs on Colab's local disk.
STATE_ROOT = Path("/content/SeqLogAD")
HF_HOME = STATE_ROOT / "hf-cache"
VENV = STATE_ROOT / "venv-py312"
RUNTIME_ROOT = STATE_ROOT / "runtime" / "seeds-ecb9a81deb2d-P2.6"
REPO_ROOT = str(RUNTIME_ROOT / "seqlogad_code_seed_confirmation")
DATA_ROOT = str(RUNTIME_ROOT / "phase2_sequence")
SEMANTIC_ROOT = str(RUNTIME_ROOT / "phase2")
GRAPH_ROOT = str(RUNTIME_ROOT / "phase2_graph")
EVIDENCE_ROOT = DRIVE_ROOT / "evidence"
WORK_ROOT = STATE_ROOT / "p2.6-work-S3407"
SESSION_ROOT = Path("/content/seqlogad_sessions") / uuid.uuid4().hex
OUTPUT_ROOT = str(SESSION_ROOT / "seqlogad_outputs")
EXPECTED_CODE_ZIP_SHA256 = "ecb9a81deb2df85e9eb2abfffd571c75ec7340de7d059cf145a4d91029d44506"
EXPECTED_DATA_ZIP_SHA256 = "4ed76b92e4b001138b28b4b3b4d3017d6ff7f30dd31e8cca0649bfdf34ede781"
EXPECTED_SEMANTIC_ZIP_SHA256 = "ebcfdfd6eb7b1607eb6d57e60fae88456d03bcb8ae2c6688ea398042c856224b"
EXPECTED_BUNDLE_SHA256 = "1268b669a8b1b474658cfa7daa6556c987b0f31f07e0101ed30af250a8d7f129"
EXPECTED_GRAPH_ZIP_SHA256 = "595b5dc7ec1b96abc80b4722b5f39dcf1c5653ec25aefebcce7ee74dbf36f9ac"
EXPECTED_GRAPH_MANIFEST_SHA256 = "8918172b4376fd822ec812e9b46b2725f83b517a43c94f3efe16f55623c9d059"
MIN_VRAM_GIB = 20

assert RUN_MODE == "development" and SEED in (3407, 8675309)
assert TARGETS == ["BGL", "HDFS"]
assert all(path.is_file() for path in FINAL_ZIPS.values()), "A FINAL ZIP is missing on Drive"
assert NOTEBOOK_STATUS == "READY", "Notebook is fail-closed: " + ", ".join(BLOCKERS)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
for directory in (STATE_ROOT, PERSISTENT_ROOT, HF_HOME, SESSION_ROOT, EVIDENCE_ROOT, WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print({"session_root": str(SESSION_ROOT), "persistent_root": str(PERSISTENT_ROOT),
       "upload_folders": [str(root) for root in UPLOAD_ROOTS]})


In [ ]:
# Colab normally includes uv; install it only if absent.
import sys, shutil, subprocess

uv_real = shutil.which("uv")
if not uv_real:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.8.17"], check=True)
    uv_real = shutil.which("uv")
assert uv_real, "uv installation failed"
uv_expected = Path(sys.executable).parent / "uv"
if not uv_expected.exists():
    uv_expected.symlink_to(uv_real)
print("uv:", uv_real)


## Verify uploads, extract atomically and build the frozen runtime


In [ ]:
from pathlib import PurePosixPath
import hashlib, json, shutil, stat, subprocess, sys, zipfile

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def locate(expected):
    # First match wins, so a ZIP copied into both upload folders is not ambiguous.
    for root in UPLOAD_ROOTS:
        for path in (sorted(root.glob("*.zip")) if root.is_dir() else []):
            if digest(path) == expected:
                return path
    raise AssertionError(f"Put the ZIP with SHA-256 {expected} in one of {[str(r) for r in UPLOAD_ROOTS]}")

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as handle:
        infos = handle.infolist()
        names = [item.filename for item in infos]
        assert len(names) == len(set(names)), "Duplicate ZIP member"
        for item in infos:
            member = PurePosixPath(item.filename)
            assert not member.is_absolute() and ".." not in member.parts and "\\" not in item.filename
            assert ((item.external_attr >> 16) & 0o170000) != stat.S_IFLNK, "ZIP symlink forbidden"
        assert handle.testzip() is None, "ZIP CRC failure"
        handle.extractall(destination)

code_zip = locate(EXPECTED_CODE_ZIP_SHA256)
data_zip = locate(EXPECTED_DATA_ZIP_SHA256)
semantic_zip = locate(EXPECTED_SEMANTIC_ZIP_SHA256)
graph_zip = locate(EXPECTED_GRAPH_ZIP_SHA256)
marker = RUNTIME_ROOT / "EXTRACTION.json"
expected_marker = {
    "code_sha256": EXPECTED_CODE_ZIP_SHA256,
    "data_sha256": EXPECTED_DATA_ZIP_SHA256,
    "semantic_sha256": EXPECTED_SEMANTIC_ZIP_SHA256,
    "graph_sha256": EXPECTED_GRAPH_ZIP_SHA256,
}
if RUNTIME_ROOT.exists():
    assert marker.is_file(), f"Unverified runtime directory exists: {RUNTIME_ROOT}"
    assert json.loads(marker.read_text()) == expected_marker, "Runtime extraction identity mismatch"
else:
    partial = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + ".partial")
    if partial.exists():
        shutil.rmtree(partial)
    partial.mkdir(parents=True)
    safe_extract(code_zip, partial)
    safe_extract(data_zip, partial)
    safe_extract(semantic_zip, partial)
    safe_extract(graph_zip, partial)
    assert (partial / "seqlogad_code_seed_confirmation").is_dir()
    assert (partial / "phase2_sequence/manifests/bundle.json").is_file()
    assert (partial / "phase2/manifests/bundle.json").is_file()
    assert (partial / "phase2_graph/manifests/graph-bundle.json").is_file()
    (partial / "EXTRACTION.json").write_text(json.dumps(expected_marker, sort_keys=True) + "\n")
    partial.rename(RUNTIME_ROOT)

assert Path(REPO_ROOT).is_dir()
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
assert digest(Path(DATA_ROOT) / "manifests/bundle.json") == EXPECTED_BUNDLE_SHA256
assert digest(Path(GRAPH_ROOT) / "manifests/graph-bundle.json") == EXPECTED_GRAPH_MANIFEST_SHA256
print({"code_zip": str(code_zip), "data_zip": str(data_zip), "semantic_zip": str(semantic_zip),
       "extraction": "PASS"})

def isolated_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    return env

def run_live(args, check=True):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=isolated_env()) as process:
        for line in process.stdout:
            token = os.environ.get("HF_TOKEN")
            if token:
                line = line.replace(token, "[TOKEN_HIDDEN]")
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, args)
    return code

run_live(["nvidia-smi"])
PYTHON = str(VENV / "bin/python")
if not Path(PYTHON).is_file():
    if sys.version_info[:2] == (3, 12):
        run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"])
        run_live([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV)])
    else:
        run_live([sys.executable, "-m", "pip", "install", "uv==0.8.17"])
        uv = str(Path(sys.executable).parent / "uv")
        assert Path(uv).is_file(), "uv bootstrap executable missing"
        run_live([uv, "python", "install", "3.12"])
        run_live([uv, "venv", "--python", "3.12", "--seed", str(VENV)])

run_live([PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"])
run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
          "https://download.pytorch.org/whl/cu128"])
run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"])
run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; install_missing(" + repr(REPO_ROOT) + ")"])
hardware_check = f"""
import json, torch
assert torch.cuda.is_available(), "CUDA is unavailable"
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info(0)
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
capability = torch.cuda.get_device_capability(0)
assert total >= {MIN_VRAM_GIB} * 1024**3, f"Expected at least {MIN_VRAM_GIB} GiB VRAM; got {{total / 1024**3:.1f}} GiB on {{name}}"
assert native_bf16, f"P2.6 scoring of P2.1/P2.2 requires native BF16 (L4/A100/H100/G4); {{name}} does not support it"
print(json.dumps({{"gpu": name, "compute_capability": capability, "free_vram_bytes": free, "total_vram_bytes": total,
                  "native_bf16": native_bf16, "torch": torch.__version__,
                  "cuda": torch.version.cuda}}, indent=2))
"""
run_live([PYTHON, "-c", hardware_check])
run_live([PYTHON, "-m", "seqlogad.sequence.cli", "environment", "--repo-root", REPO_ROOT])


## Secure Hugging Face authentication


In [ ]:
# Read the token from Colab Secrets; it is never printed or written to Drive.
from google.colab import userdata

token = os.environ.get("HF_TOKEN")
if not token:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
assert token, (
    "Add HF_TOKEN in Colab Secrets, enable notebook access, then rerun this cell"
)
run_live([PYTHON, "-m", "seqlogad.sequence.cli", "auth", "--repo-root", REPO_ROOT])


## Sequential target loop with persistent recovery

In [ ]:
# Scores the evaluation windows; one JSON progress line per 64 windows.
args = [
    PYTHON, "-m", "seqlogad.seeds.score",
    "--seed", str(SEED),
    "--repo-root", REPO_ROOT,
    "--data-root", DATA_ROOT,
    "--semantic-root", SEMANTIC_ROOT,
    "--graph-root", GRAPH_ROOT,
    "--output-root", str(EVIDENCE_ROOT),
    "--work-root", str(WORK_ROOT),
    "--tasks", SCORING_TASKS,
]
for task, path in FINAL_ZIPS.items():
    args += ["--final", f"{task}={path}"]
run_live(args, check=True)


## Final persistent-integrity and safe-stop gate


In [ ]:
# Every expert and fold must have a complete evidence file and receipt; package them for download.
experts = {"P2.1": "SEMANTIC_LLAMA", "P2.2": "SEQUENCE_LLAMA_REFERENCE", "P2.3": "SEQUENCE_LIGHTWEIGHT",
           "P2.4": "GTAT_STRUCTURAL_TEMPORAL"}
package = DRIVE_ROOT / f"P2.6_evidence_S{SEED}.zip"
with zipfile.ZipFile(SESSION_ROOT / package.name, "w", zipfile.ZIP_DEFLATED) as handle:
    for target in TARGETS:
        fold = f"FOLD-TARGET-ARCH-{target}"
        for task, expert in experts.items():
            evidence = EVIDENCE_ROOT / fold / f"{expert}.jsonl"
            receipt = json.loads((EVIDENCE_ROOT / fold / f"{expert}.receipt.json").read_text())
            assert receipt["evidence_sha256"] == digest(evidence), f"{task} {target} evidence changed"
            assert receipt["sanity_reproduction"]["pass"] and not receipt["labels_read"]
            handle.write(evidence, f"evidence/{fold}/{evidence.name}")
            handle.write(evidence.with_suffix(".receipt.json"), f"evidence/{fold}/{expert}.receipt.json")
            print(task, target, receipt["windows"], receipt["status_counts"])
shutil.copyfile(SESSION_ROOT / package.name, package)
package_hash = digest(package)
Path(str(package) + ".sha256").write_text(f"{package_hash}  {package.name}\n")
shutil.rmtree(SESSION_ROOT)
print("====================================")
print("SEQLOGAD COLAB EXECUTION SUMMARY - P2.6 SCORING PASS")
print(f"EVIDENCE_ZIP={package}")
print(f"EVIDENCE_ZIP_SHA256={package_hash}")
print("PERSISTENT_STORAGE_VERIFIED=YES")
print("SAFE_TO_DISCONNECT_COLAB=YES")
print("====================================")
